# C389 CA1: try the trial model (deployment)

Loads the weights from `CA1_train_trial.ipynb` and serves the team's app: counts people on foot and wheeled crossers moving left and right, and draws the boxes.

Two ways to try photos:
- **Section 4:** upload photos straight into this notebook and see the result below the cell.
- **Section 5:** open the Gradio page through a public link (the same page used for the demo).

The app code comes from the team repo (`app/service.py`, `src/counting.py`), so what you try here is what ships.

**Before committing this notebook:** *Edit → Clear all outputs*. Outputs can hold photos of people.

## 1. Settings

The repo is private, so cloning needs a GitHub token. Add one in Colab's **Secrets** panel (key icon on the left) named `GITHUB_TOKEN`, with notebook access switched on. A fine-grained token with read-only *Contents* on `c389-ca1` is enough. With no secret, the next cell asks for it.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/C389_Supernova_Model"
RUN_NAME   = "final_4cls_ig_v2"
WEIGHTS    = f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt"

REPO   = "Morn-Thuta/c389-ca1"
BRANCH = "feat/wheeled-classes"   # has the 4-class app code until the PRs are merged; then use "main"
ULTRALYTICS_VERSION = "8.4.120"

## 2. Install, get the code, connect Drive

In [ ]:
import os
import subprocess
from getpass import getpass

subprocess.run(["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio", "imagehash"], check=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("GitHub token (read access to %s): " % REPO)

if not os.path.isdir("/content/c389-ca1"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://x-access-token:%s@github.com/%s.git" % (token, REPO),
                    "/content/c389-ca1"], check=True)
else:
    subprocess.run(["git", "-C", "/content/c389-ca1", "pull", "-q"], check=True)
del token
os.chdir("/content/c389-ca1")
print("code from branch", BRANCH)

from google.colab import drive
drive.mount("/content/drive")
if not os.path.isfile(WEIGHTS):
    raise FileNotFoundError(f"No weights at {WEIGHTS}. Run the training notebook first, or fix RUN_NAME.")
print("weights:", WEIGHTS)

## 3. Load the model once

With the 4-class weights the mode reads "twist (people + wheeled)". The 2-class baseline reads "baseline (people only)".

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", stream=sys.stdout, force=True)

from app import service
from app.app import build_demo
from src.config import CONF_THRESHOLD

DETECTOR = service.load_model(WEIGHTS)
print("mode   :", DETECTOR.mode)
print("classes:", DETECTOR.class_names)
print("default confidence threshold:", CONF_THRESHOLD)

## 4. Try your own photos here

Run the cell, click **Choose Files**, and pick one or more photos. Each one is shown with its boxes and counts.

- people: red = moving left, blue = moving right
- wheeled crossers (thick box): orange = moving left, purple = moving right
- Change `THRESHOLD` to see what a stricter or looser model finds.

In [ ]:
from google.colab import files
from IPython.display import display

THRESHOLD = CONF_THRESHOLD

uploaded = files.upload()
for name, data in uploaded.items():
    picture, reply = service.predict(data, THRESHOLD)
    print("=" * 60)
    print(name)
    if picture is None:
        print("refused:", reply["status"], reply["error"], "-", reply.get("detail", ""))
        continue
    counts = reply["counts"]
    print("people   left %(person_left)d  right %(person_right)d" % counts)
    print("wheeled  left %(wheeled_left)d  right %(wheeled_right)d" % counts)
    print("total    left %(total_left)d  right %(total_right)d" % counts)
    print("time: %.2f s" % reply["inference_seconds"])
    picture = picture.copy()
    picture.thumbnail((1200, 1200))
    display(picture)

## 5. Open the Gradio page (public link)

`share=True` prints a `gradio.live` link that anyone holding it can open, until this runtime stops (at most a week). Share it only with the team, and run section 6 when finished.

In [ ]:
demo = build_demo()
demo.launch(share=True)

## 6. Close the page

In [ ]:
demo.close()
print("The share link is closed.")